In [ ]:
!python -m pip install duckdb

# Test Code

In [ ]:
import duckdb

duckdb.sql("SELECT 'whistling_duck' AS waterfowl, 'whistle' AS call")

# Parquet Import Code

In [ ]:
import duckdb

conn = duckdb.connect(database="presidents.db")

presidents_relation = conn.read_parquet("presidents.parquet")

conn.sql(
    """
    SELECT sequence, last_name, first_name
    FROM presidents_relation
    WHERE sequence <= 2
    """
).show()

presidents_relation.to_table("presidents")

conn.close()

In [ ]:
with duckdb.connect(database="presidents.db") as conn:
    conn.sql(
        """
        SELECT last_name, first_name
        FROM presidents
        WHERE last_name = 'Washington' 
        """
    ).show()

# Data Interpretation

In [ ]:
import duckdb

with duckdb.connect(database="presidents.db") as conn:
    presidents_relation = conn.read_csv("presidents.csv")
    presidents_relation.limit(2)

In [ ]:
import duckdb

with duckdb.connect(database="presidents.db") as conn:
    presidents_relation = conn.read_csv(
        "presidents.csv", date_format="%B %d %Y"
    )
    presidents_relation.dtypes

# Database Queries

In [ ]:
import duckdb

with duckdb.connect(database="presidents.db") as conn:
    (conn.read_json("parties.json").to_table("parties"))

with duckdb.connect("presidents.db") as conn:
    conn.sql(
        """
        SELECT first_name, last_name, party_name
        FROM parties
        JOIN presidents
        ON presidents.party_id = parties.party_id
        ORDER BY last_name DESC
        """
    ).show()

In [ ]:
import duckdb

presidents = duckdb.read_parquet("presidents.parquet")
parties = duckdb.read_json("parties.json")

duckdb.sql(
    """
    SELECT first_name, last_name, party_name
    FROM parties
    JOIN presidents
    ON presidents.party_id = parties.party_id
    ORDER BY last_name DESC
    """
)

In [ ]:
presidents = duckdb.read_parquet("presidents.parquet").set_alias("presidents")
parties = duckdb.read_json("parties.json").set_alias("parties")

(
    presidents.join(parties, "presidents.party_id = parties.party_id")
    .select("first_name", "last_name", "party_name")
    .order("last_name DESC")
)

# Concurrent Access

In [ ]:
import concurrent.futures
import duckdb


def read_data(thread_id):
    print(f"Thread {thread_id} starting its read.")
    with duckdb.connect("presidents.db") as conn:
        conn.sql(
            """
            SELECT first_name, last_name
            FROM presidents
            WHERE sequence = 1
            """
        ).show()
    print(f"Thread {thread_id} ending its read.")


def concurrent_access():
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        executor.map(read_data, range(3))


concurrent_access()

In [ ]:
import concurrent.futures
import duckdb


def update_data(thread_id):
    new_name = f"George ({thread_id})"
    with duckdb.connect("presidents.db") as conn:
        print(f"Thread {thread_id} starting update.")
        conn.sql(
            f"""
            UPDATE presidents
            SET first_name = '{new_name}'
            WHERE sequence = 1
            """
        )
        print(f"Thread {thread_id} ending update.")


def concurrent_access():
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        executor.map(update_data, range(3))


concurrent_access()

In [ ]:
with duckdb.connect("presidents.db") as conn:
    conn.sql(
        """
            SELECT last_name, first_name
            FROM presidents
            WHERE sequence = 1
            """
    )

## Integrating With Python's Environment

In [ ]:
import duckdb


def short_name(first_name: str, last_name: str) -> str:
    return f"{first_name[0]}. {last_name}"


duckdb.remove_function("short_name")  # Prevents NotImplementedException
duckdb.create_function("short_name", short_name)


presidents = duckdb.read_parquet("presidents.parquet")

duckdb.sql(
    """ 
    SELECT short_name(first_name, last_name) AS name,
    (term_end - term_start) AS "days in office"
    FROM presidents
    """
).limit(3)

In [ ]:
python -m pip install pandas polars

In [ ]:
import duckdb
import pandas

with duckdb.connect("presidents.db") as conn:
    pandas_df = conn.sql(
        """
        SELECT last_name, first_name
        FROM presidents
        WHERE sequence <= 4
        """
    ).df()

pandas_df

In [ ]:
import polars

presidents = duckdb.read_parquet("presidents.parquet").set_alias("presidents")
parties = duckdb.read_json("parties.json").set_alias("parties")

(
    presidents.join(parties, "presidents.party_id = parties.party_id")
    .select("first_name", "last_name", "party_name")
    .order("last_name DESC")
).pl().head(3)